# AVA 云端算力工作站（Google Colab 一键自举版）

本笔记本供 AVA Agent（`cloud up --provider colab` 自动拉起）与人工复用：
- **单 Cell 一键执行**：挂载 Drive 缓存 → 秒级恢复 `ava_venv` → 拉取最新代码 → 启动 `sshd` + `cloudflared` 隧道（或 `drive` 队列）→ 输出 `AVA_COLAB_SSH_READY:` 信标。
- **卡型建议**：`tts` / `asr` 选 **T4 (16GB)** 即可；`captions`（Qwen3-VL-8B）请选 **A100**（L4 实测可用显存 22.5GB，VLM 可能紧张）。
- **节省 CU**：内置 `run_ssh_sentinel` / `run_drive_loop`，收到 `cloud down` 或空闲超 `IDLE_SECONDS` 自动调用 `google.colab.runtime.unassign()`。

In [ ]:
#@title 一键启动 AVA Colab 运行时（恢复环境 + 拉代码 + 开隧道/队列 + 空闲自动释放 CU）
import os, re, shutil, subprocess, sys, time
from pathlib import Path

MODE = "ssh" #@param ["ssh", "drive"]
REPO_URL = "https://github.com/songhuiming2007-coder/anime-video-agent.git" #@param {type:"string"}
REPO_BRANCH = "main" #@param {type:"string"}
TASK_PROFILE = "none" #@param ["none", "tts", "asr", "captions", "all"]
IDLE_SECONDS = 300 #@param {type:"integer"}
SSH_PUBLIC_KEY = "" #@param {type:"string"}
DRIVE_QUEUE_DIR = "/content/drive/MyDrive/ava_cloud" #@param {type:"string"}
FORCE_REINSTALL_VENV = False #@param {type:"boolean"}

REPO_DIR = "/content/anime-video-agent"
MODELS_ROOT = "/content/models"
DRIVE_CACHE = "/content/drive/MyDrive/ava_cache"

def run(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    return subprocess.run(["bash", "-lc", cmd], check=check)

def sh(cmd, check=True):
    return subprocess.run(["bash", "-lc", cmd], check=check, capture_output=True, text=True)

# 1. 挂载 Drive 缓存与工作目录
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
Path(DRIVE_CACHE).mkdir(parents=True, exist_ok=True)
Path(MODELS_ROOT).mkdir(parents=True, exist_ok=True)

# 2. 拉取或更新仓库
if not Path(f"{REPO_DIR}/.git").is_dir():
    run(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    run(f"git -C {REPO_DIR} fetch origin {REPO_BRANCH} --depth 1 && git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}")

# 3. 恢复或构建 /content/venv（按 Python 主次版本命中 Drive 缓存）
venv_dir = Path("/content/venv")
py_tag = f"py{sys.version_info.major}{sys.version_info.minor}"
venv_tar = Path(DRIVE_CACHE) / f"ava_venv_{py_tag}.tar.gz"
if FORCE_REINSTALL_VENV and venv_dir.exists():
    shutil.rmtree(venv_dir)

if not (venv_dir / "bin" / "python").exists():
    if venv_tar.is_file() and not FORCE_REINSTALL_VENV:
        print(f"[*] 从 Drive 缓存恢复 venv: {venv_tar}", flush=True)
        run(f"mkdir -p /content && tar -xzf {venv_tar} -C /content")
    if not (venv_dir / "bin" / "python").exists():
        run("pip install -q uv")
        run(f"uv venv --system-site-packages --python {sys.executable} {venv_dir}")
        run(
            f"cd {REPO_DIR} && uv pip install --python {venv_dir}/bin/python "
            "-e . qwen-tts funasr torch_complex 'transformers>=4.57' accelerate soundfile scipy huggingface_hub hf_transfer"
        )
        print(f"[*] 备份 venv 到 Drive 缓存以便下次秒开: {venv_tar}", flush=True)
        run(f"tar -czf {venv_tar}.tmp -C /content venv && mv {venv_tar}.tmp {venv_tar}", check=False)

run(f"{venv_dir}/bin/python -c 'import torch, qwen_tts, transformers; from funasr import AutoModel; print(\"[OK] CUDA:\", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\")'")

# 4. 可选：按需预下载模型（默认 none，运行时从 HF 直拉）
PROFILE_MODELS = {
    "none": [],
    "tts": ["Qwen/Qwen3-TTS-12Hz-1.7B-Base", "openai/whisper-large-v3", "FunAudioLLM/SenseVoiceSmall"],
    "asr": ["openai/whisper-large-v3", "FunAudioLLM/SenseVoiceSmall"],
    "captions": ["Qwen/Qwen3-VL-8B-Instruct"],
    "all": [
        "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        "Qwen/Qwen3-VL-8B-Instruct",
        "BAAI/bge-m3",
        "openai/whisper-large-v3",
        "FunAudioLLM/SenseVoiceSmall",
    ],
}
models_to_pull = PROFILE_MODELS[TASK_PROFILE]
if models_to_pull:
    dl_code = (
        "import os; from huggingface_hub import snapshot_download; "
        f"models = {models_to_pull!r}; "
        f"root = {MODELS_ROOT!r}; "
        "[print('[HF]', m) or snapshot_download(m, local_dir=f'{root}/{m}') for m in models]"
    )
    run(f"HF_HUB_ENABLE_HF_TRANSFER=1 {venv_dir}/bin/python -c {dl_code!r}")

# 5. 启动 SSH 隧道或 Drive 共享队列，并进入空闲/down 自动释放看守
sys.path.insert(0, REPO_DIR)
from tools import colab_worker

if MODE == "ssh":
    sh("DEBIAN_FRONTEND=noninteractive apt-get update -qq && apt-get install -y -qq openssh-server tmux rsync curl")
    if not shutil.which("cloudflared"):
        sh("curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
    ssh_dir = Path("/root/.ssh")
    ssh_dir.mkdir(mode=0o700, exist_ok=True)
    ak = ssh_dir / "authorized_keys"
    drive_ak = Path(DRIVE_CACHE) / "authorized_keys"
    pub = SSH_PUBLIC_KEY.strip() or (drive_ak.read_text(encoding="utf-8").strip() if drive_ak.is_file() else "")
    if not pub:
        raise RuntimeError(f"请填写 SSH_PUBLIC_KEY 或把本机公钥放到 {drive_ak}")
    if SSH_PUBLIC_KEY.strip() and not drive_ak.is_file():
        drive_ak.write_text(pub + "\n", encoding="utf-8")
    ak.write_text(pub + "\n", encoding="utf-8")
    ak.chmod(0o600)
    sh("mkdir -p /var/run/sshd && ssh-keygen -A && sed -i 's/#PubkeyAuthentication yes/PubkeyAuthentication yes/; s/#PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config && service ssh restart")
    port_res = sh("grep -E '^Port[[:space:]]+' /etc/ssh/sshd_config | awk '{print $2}' | head -n 1", check=False)
    ssh_port = int(port_res.stdout.strip()) if port_res.stdout.strip().isdigit() else 22
    cf_log = Path("/tmp/cloudflared.log")
    if cf_log.exists():
        cf_log.unlink()
    subprocess.Popen(["cloudflared", "tunnel", "--url", f"ssh://127.0.0.1:{ssh_port}", "--logfile", str(cf_log)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    tunnel_host = None
    for _ in range(30):
        if cf_log.is_file():
            m = re.search(r"https://([A-Za-z0-9-]+\.trycloudflare\.com)", cf_log.read_text(errors="replace"))
            if m:
                tunnel_host = m.group(1)
                break
        time.sleep(1)
    if not tunnel_host:
        raise RuntimeError(f"未获取 trycloudflare 域名，请查看 {cf_log}")
    print("=" * 70, flush=True)
    print(f"AVA_COLAB_SSH_READY: root@{tunnel_host}", flush=True)
    print(f"python -m pipeline.cloud connect {tunnel_host} --provider colab", flush=True)
    print("=" * 70, flush=True)
    colab_worker.run_ssh_sentinel(idle_seconds=int(IDLE_SECONDS), unassign_on_exit=True)
else:
    print("=" * 70, flush=True)
    print(f"AVA_COLAB_DRIVE_READY: {DRIVE_QUEUE_DIR}", flush=True)
    print("=" * 70, flush=True)
    colab_worker.run_drive_loop(
        Path(DRIVE_QUEUE_DIR),
        Path(REPO_DIR),
        "/content/venv/bin/python",
        Path(MODELS_ROOT),
        idle_seconds=int(IDLE_SECONDS),
        unassign_on_exit=True,
    )